In [9]:
# Imports
from pdf2image import convert_from_path
from pdf2image.exceptions import ( PDFInfoNotInstalledError,  PDFPageCountError,  PDFSyntaxError)
from pdfminer.high_level import extract_text
import base64
import io
import os
import concurrent.futures
from tqdm import tqdm
# from openai import OpenAI
import re
import pandas as pd 
from sklearn.metrics.pairwise import cosine_similarity
import json
import numpy as np
from rich import print as richprint
from rich.console import Console # Console 임포트 추가
from ast import literal_eval
import unicodedata
import difflib

In [10]:
from pdfminer.high_level import extract_text
from tqdm import tqdm
import re


def make_buffer_page(page):
    new_page = page[:]
    for i in range(len(page)):
        if not page[i]:
            left = page[i - 1] if i > 0 else False
            right = page[i + 1] if i < len(page) - 1 else False
            if left or right:
                new_page[i] = True
    return new_page



def pdf_Rf_page_img(pdf_list):
    Rf_in_multi_paper = []
    for i, pdf in enumerate(pdf_list):
        print(f'pdf {i}')
        text = extract_text(pdf)
        if '\x0c' in text:

            rf_pages_bool = [('Rf' in  txt_page or 'RF' in txt_page) for txt_page in text.split('\x0c')]
            rf_pages_bool_buffer = make_buffer_page(rf_pages_bool)
            rf_pages_num = np.arange(0,len(rf_pages_bool),1)[rf_pages_bool_buffer]

            pages_text = [txt_page for txt_page in text.split('\x0c')]
            rf_pages_text = [pages_text[i] for i in rf_pages_num]
            rf_pages_united_string = '\x0c'.join(rf_pages_text)

            sen_rf_mnch_in_pdf=[]
            print("Checking each page...")
            for txt in tqdm(rf_pages_text):
                sen_list = txt.split('\n')
                sen_list = [sen for sen in sen_list if not re.fullmatch(r'\s*', sen)]
                sen_idx = [[idx-1, idx, idx+1, idx+2] for idx, sen in enumerate(sen_list) if ('Rf' in  sen or 'RF' in sen)]
                sen_rf_mnch_in_page=[]
                for mnch in sen_idx:
                    sen_rf_mnch_in_page.append([sen_list[i] for i in mnch if (i>=0)&(i<len(sen_list))])
                sen_rf_mnch_in_pdf.append(sen_rf_mnch_in_page)
                
            
        else:
            print(f'{pdf} : 페이지 표시자(\x0c)가 없습니다!!')
            continue

        imgs = convert_from_path( pdf, poppler_path = '/home/ufslab223/anaconda3/envs/ksw/bin')
        rf_page_imgs = [imgs[i] for i in rf_pages_num]
        Rf_in_multi_paper.append(rf_page_imgs)
    return Rf_in_multi_paper, rf_pages_united_string, rf_pages_text, sen_rf_mnch_in_pdf



# Converting images to base64 encoded images in a data URI format to use with the ChatCompletions API
# 이미지 이진 데이터-->텍스트 문자열 (base64)
# URI : 파일 내용을 문자열로 표현한 URL 형식

def get_img_uri(img):
    png_buffer = io.BytesIO()
    img.save(png_buffer, format="PNG")
    png_buffer.seek(0)

    base64_png = base64.b64encode(png_buffer.read()).decode('utf-8')

    data_uri = f"data:image/png;base64,{base64_png}"
    return data_uri

def get_img_markdown(img):
    
    uri = get_img_uri(img)
    markdown = f'![image](data:image/png;base64,{uri})\n\nDescribe this image.'

# GPT-API (one page each)

### To use this notebook, create your own OpenAI API key and insert it into the `api_key=''` field in the code.

In [ ]:
#GPT-API
import openai
from openai import OpenAI
import time

def safe_chat(client, **kwargs):
    # 분당 토큰 처리 일정 이상 넘어가면 에러 뜨는데, 몇초 기다리면 제대로 됨. 
    # 중간에 안터지게 몇초 기다리고 재시도 하게 하는 함수
    for _ in range(5): # 5변 시도
        try:
            return client.chat.completions.create(**kwargs)
        except openai.RateLimitError as e:
            print(" 분당 토큰 수 제한 때문에... 잠깐만 기달려봐...")
            time.sleep(1.5) # 1.5초 기다렸다가 재시도
    raise RuntimeError("분당 토큰 수 제한이 계속 걸리네... 나중에 다시 시도해봐")


def analyze_one_page_GPT(one_page_text):

    client = OpenAI(api_key='')

    json_style_example='''
    [
      {
        "compound name": "26b",
        "compound IUPAC name": "the given IUPAC name",
        "eluent1": "PE",
        "eluent2": "EA",
        "ratio": "20/1",
        "Rf": "0.6"
      },
      {
        "compound name": "26c",
        "compound IUPAC name": "another IUPAC name",
        "eluent1": "CH2Cl2",
        "eluent2": "np.nan",
        "ratio": "1",
        "Rf": "0.4"
      }
    ]
    '''
    print('GPT started to analyze pdf')
    answer=[]
    for i in tqdm(range(len(one_page_text))):
        text_input = [{"type": "text", "text": one_page_text[i]}]
        #image_input = [{"type": "image_url", "image_url": {"url": one_page_image[i]}}]

        if i==0:
            supporting_page_text = "This is the first page, so there is no previous page."
        else:
            supporting_page_text = one_page_text[i-1]

        if len(answer)>0:
            clean_answer = answer[-1].strip().removeprefix("```json").removesuffix("```").strip()
            previous_answer = json.loads(clean_answer)
        else:
            previous_answer=[]

        system_prompt = f'''
        You will be provided with the text and image of a PDF page.
        
        Check whether each compound on the page meets the following 4 conditions:
        1. The text includes a compound with an Rf value specified.
        2. The compound with the Rf value must has its IUPAC name explicitly written. If the IUPAC name is not found or is ambiguous, **you must completely ignore the compound and do not include it in the output**.
        3. The eluent used to measure the Rf value is specified. It must be either a single solvent or a mixture of two solvents.
        4. The ratio between the solvents (if it is a mixture) is clearly written.
        
        For every compound that satisfies **all four** conditions, extract the information and return it as a **list of Python-style dictionaries** (JSON format), where **all values are strings**.
        You must ignore any compound that does not satisfy all four conditions completely. Do not attempt to guess or fill in any missing field.

        Note: The information about the Rf value for a compound always appears **after** the compound's name in the text. Always associate the Rf value with the correct compound whose name precedes the Rf information.
   

        - If both TLC and column chromatography solvents are mentioned, 
          you **must use the solvent composition associated with the Rf value measurement**.
        
        - You can identify this because the Rf value (e.g., "Rf = 0.5") will often be followed by or associated with a solvent composition (e.g., "Rf = 0.5 (30% EtOAc in hexane)").
        
        - ⚠️ Do NOT use the solvent ratio used for purification or flash chromatography unless it is explicitly mentioned as the condition for Rf measurement.

        
        Special handling:
        - If only **one eluent** is used, assign its name to `"eluent1"`, set `"eluent2"` to `"np.nan"`, and set `"ratio"` to `"1"` (as a string).
        - If two eluents are used, assign each to `"eluent1"` and `"eluent2"` respectively, and extract the written ratio (e.g., `"20:1"`).
        
        - When expressing the `"ratio"`, follow these conversion rules:
            • If written as `"30% eluent1 in eluent2"`, convert to `"30/70"`
            • If written as `"eluent1 / eluent2 = 30/1"`, keep as `"30/1"`
            • If written as `"eluent1 / eluent2 = 20:1"`, convert to `"20/1"`
        
        Output format example:
        {json_style_example}
       
        
        ⚠️ Do not include any other text, explanations, or formatting.  
        Only return the list of dictionaries exactly as shown above.

        Do not include data that were already reported on the previous page, even if they appear again. 
        Only report **new, previously unreported data** from the current page.
        Already reported data: {previous_answer}

        
        The following is the previous page for contextual reference. Refer to it if the current page lacks information:
        {supporting_page_text} 
        '''
        
        response = safe_chat(client,
                             model="gpt-4.1-mini",
                             messages=[
                                 {"role": "system", "content": system_prompt},
                                 {"role": "user", "content": text_input}# + image_input},
                             ],
                             temperature=0,
                             top_p=0.1,
                             #reasoning_effort='high'
                            )
        answer.append(response.choices[0].message.content)
        
    return answer


# collect Rf pages

In [12]:
def collect_rf_pages(pdf_file):
    print('preprocessing pdf')
    # pdf_file example : 'ol4c04571_si_001.pdf'

    # extract
    path = './SI/'
    pdf_list= [path+pdf_file]
    rf_multi_paper, rf_multi_string, short_text, sen_rf_mnch_in_pdf = pdf_Rf_page_img(pdf_list)

    #image --> url
    rf_url_multi_paper = []
    for rf_one_paper in rf_multi_paper:
        rf_page_url = [get_img_uri(rf_page) for rf_page in rf_one_paper]
        rf_url_multi_paper.append(rf_page_url)
        
    return rf_multi_string, short_text,  sen_rf_mnch_in_pdf, rf_url_multi_paper

# extract data (GPT)

In [13]:
def normalize_ligatures(s):
    return unicodedata.normalize('NFKC', s)

def most_similar_substring(text, target, mode = 'IUPAC_string', starting_index = 0):
    text =  normalize_ligatures(re.sub(r'\s+','',text))
    best_match = ""
    highest_ratio = 0
    best_index = -1
    if type(target)==str:
        target_len = len(target)
        for i in range(starting_index, len(text) - target_len + 1):
            if (i + target_len)>len(text):
                print('check IUPAC. similar IUPAC is out of range')
                best_index = len(text)
                break
            chunk = text[i:i+target_len]
            ratio = difflib.SequenceMatcher(None, chunk, target).ratio()
            if ratio > highest_ratio:
                highest_ratio = ratio
                best_match = chunk
                best_index = i
    if mode == 'IUPAC_string':            
        return best_match.strip()
    elif mode == 'IUPAC_index':
        return best_index

def most_similar_substring_limited_idx(text, target, ending_index, mode = 'IUPAC_string',  starting_index = 0):
    text =  normalize_ligatures(re.sub(r'\s+','',text))
    best_match = ""
    highest_ratio = 0
    if type(target)==str:
        target_len = len(target)
        for i in range(starting_index, ending_index):
            if (i + target_len)>len(text):
                print('check IUPAC. similar IUPAC is out of range')
                best_index = len(text)
                break
            chunk = text[i:i+target_len]
            ratio = difflib.SequenceMatcher(None, chunk, target).ratio()
            if ratio > highest_ratio:
                highest_ratio = ratio
                best_match = chunk
                best_index = i
    if mode == 'IUPAC_string':            
        return best_match.strip()
    elif mode == 'IUPAC_index':
        return best_index


def false_iupac_compare(short_text, data_final, false_page_idx):
    page_idx_list = list(false_page_idx.items())
    for pi_pair in page_idx_list:
        page_text = short_text[pi_pair[0]]
        if pi_pair[0] > 0:
            pre_page_text = short_text[pi_pair[0]-1]
        iupac_list = data_final.loc[pi_pair[1], 'compound IUPAC name'].tolist()
        for i, iupac in enumerate(iupac_list):
            iupac_in_paper = most_similar_substring(page_text, iupac)
            richprint(f'[bold red]index___{pi_pair[1][i]}[/bold red]')
            richprint(f'[bold red]name: {data_final.loc[pi_pair[1][i], 'compound IUPAC name']}[/bold red]')
            if type(iupac)==str:
                print('LLM_output >>>>  ', re.sub(r'\s+','',iupac))
            else:
                print('LLM_output >>>>  ', iupac)
            print('TXT_current >>>  ', iupac_in_paper)
            if pi_pair[0] > 0:
                iupac_in_paper_pre = most_similar_substring(pre_page_text, iupac)
                print('TXT_previous >>  ', iupac_in_paper_pre,'\n TXT_current가 전혀 비슷하지 않을때 참고')
            

def extract_data_GPT(pdf_file):
    import json
    whole_string, short_text, sen_rf_mnch_in_pdf, image_url_list = collect_rf_pages(pdf_file)
    image = image_url_list[0]
    if len(short_text)<=1:
        print('Cannot detect Rf pages!!!')
        return None
    extract_output1 = analyze_one_page_GPT(one_page_text = short_text)
    false_page_idx = {}
    whole_data=[]
    df_idx_start=0
    for idx, data in enumerate(extract_output1):
        clean_data = data.strip().removeprefix("```json").removesuffix("```").strip()
        dict_data = json.loads(clean_data)

        # data double check
        
        # --> Rf data check
        
        if len(dict_data)==0:
            print('No Rf data in this page')
        else:
            df_data = pd.DataFrame(dict_data)
            richprint(f'[bold bright_red]Page ---#################################---{idx}[/bold bright_red]')
            df = df_data.loc[:,['compound IUPAC name','eluent1','eluent2','ratio','Rf']]
            num = len(df)
            df.index = np.arange(df_idx_start, df_idx_start + num, 1)
            df_idx_start += num
            display(pd.DataFrame(df).loc[:,['compound IUPAC name','eluent1','eluent2','ratio','Rf']])


            # --> iupac name & matching with Rf data check

            short_text_flat = normalize_ligatures(re.sub(r'\s+','',short_text[idx]).lower())
            page_iupac_index = [short_text_flat.find(re.sub(r'\s+','',normalize_ligatures(str(iupac))).lower()) for iupac in df_data['compound IUPAC name']]
            
            for_check = page_iupac_index.copy()
            if (for_check[0] == -1)&(idx>0):
                previous_short_text_flat = normalize_ligatures(re.sub(r'\s+','',short_text[idx-1]).lower())
                pre_page_iupac_index = [previous_short_text_flat.find(re.sub(r'\s+','',normalize_ligatures(str(iupac))).lower()) for iupac in df_data['compound IUPAC name']]
                where = np.where(np.array(pre_page_iupac_index)!= -1)[0]
                for i in where:
                    max_idx = len(previous_short_text_flat)
                    minus_index = pre_page_iupac_index[i]-(max_idx+100)
                    for_check[i] = minus_index
                    page_iupac_index[i] = minus_index
                
            if -1 in for_check:
                minus_one_idx = np.where(np.array(for_check)==-1)[0]
                false_index = minus_one_idx + (df_idx_start-num)
                print(f'wrong iupac in this page:{false_index}')
                #print(num)
                false_page_idx[idx] = false_index

                for i in minus_one_idx:
                    query_iupac = df_data.loc[i,'compound IUPAC name']

                    left = 0
                    right = len(short_text_flat) - len(str(query_iupac)) + 1

                    left_candidates = [idx_val for idx_val in for_check[:i] if idx_val != -1]
                    if left_candidates:
                        left = max(left_candidates)

                    right_candidates = [idx_val for idx_val in for_check[i+1:] if idx_val != -1]
                    if right_candidates:
                        right = min(right_candidates) - len(str(query_iupac))

                    left = max(left, 0)
                    right = max(right, left + 1)

                    iupac_similar_index = most_similar_substring_limited_idx( short_text_flat,query_iupac,ending_index=right,mode='IUPAC_index',starting_index=left)

                    for_check[i] = iupac_similar_index


            ######################################## 여기서부터 변경된 이중 점검 인터페이스
            
            # 1. 인덱스 점검: 이전 페이지의 텍스트가 포함되어야 하는지 확인 (음수 인덱스 존재 여부)
            include_previous = any(idx_val < 0 for idx_val in for_check)
            
            # 이전 페이지가 필요할 경우 합쳐서 full_text 생성
            if include_previous and idx > 0:
                full_text = short_text[idx-1] + "\n\n--- [Current Page] ---\n\n" + short_text[idx]
            else:
                full_text = short_text[idx]

            # 2. IUPAC name에 대응하는 공백이 없는 가장 유사한 chunk들 추출
            short_text_flat_2 = normalize_ligatures(re.sub(r'\s+', '', short_text[idx]))
            if idx > 0:
                previous_short_text_flat_2 = normalize_ligatures(re.sub(r'\s+', '', short_text[idx-1]))
                pre_full_len = len(previous_short_text_flat_2)
            else:
                previous_short_text_flat_2 = ""
                pre_full_len = 0

            iupac_chunks = []
            for i, iupac_idx in enumerate(for_check):
                iupac_length = len(df_data.loc[i, 'compound IUPAC name']) if pd.notna(df_data.loc[i,'compound IUPAC name']) else 0
                if iupac_length == 0:
                    continue
                
                # 이전 페이지에 있는 경우
                if iupac_idx < 0:
                    org_idx = iupac_idx + (pre_full_len + 100)
                    if org_idx < 0: org_idx = 0  # 안전 장치
                    iupac_label = previous_short_text_flat_2[org_idx:(org_idx+iupac_length)]
                # 현재 페이지에 있는 경우
                else:
                    iupac_label = short_text_flat_2[iupac_idx:(iupac_idx+iupac_length)]
                
                if iupac_label:
                    iupac_chunks.append(iupac_label)

            # 3. 찾은 IUPAC chunk들을 정규식을 통해 원문에서 빨간색으로 표시하기 위해 특수 문자 플레이스홀더 사용
            placeholder_map = {}
            # 부분 문자열 치환 시 겹치는 문제를 막기 위해 문자열이 긴 순서대로 먼저 치환하도록 정렬 및 중복 제거
            iupac_chunks = sorted(list(set(iupac_chunks)), key=len, reverse=True)
            
            for i, chunk in enumerate(iupac_chunks):
                # 공백과 줄바꿈을 무시하고 매칭할 수 있도록 패턴 생성
                escaped_chars = [re.escape(c) for c in chunk]
                pattern = r'\s*'.join(escaped_chars)
                placeholder = f"__IUPAC_PLACEHOLDER_{i}__"
                try:
                    # 매칭된 부분을 나중에 복구할 수 있도록 튜플로 저장
                    def repl(match, p=placeholder):
                        placeholder_map[p] = match.group(1)
                        return p
                    full_text = re.sub(f'({pattern})', repl, full_text, flags=re.IGNORECASE)
                except Exception:
                    pass
            
            # 4. Rf가 포함된 줄을 찾아 초록색으로 표시
            lines = full_text.split('\n')
            colored_lines = []
            for line in lines:
                # Rf나 RF가 포함되어 있으면 줄 전체를 초록색 플레이스홀더로 감쌈
                if 'Rf' in line or 'RF' in line:
                    colored_lines.append(f"__GREEN_START__{line}__GREEN_END__")
                else:
                    colored_lines.append(line)
            
            colored_text = '\n'.join(colored_lines)

            # 5. 플레이스홀더를 실제 Rich 태그로 치환
            for placeholder, original_text in placeholder_map.items():
                colored_text = colored_text.replace(placeholder, f"[bold orange1]{original_text}[/bold orange1]")
            
            colored_text = colored_text.replace("__GREEN_START__", "[bold green1]").replace("__GREEN_END__", "[/bold green1]")

            # 숫자가 파란색 등으로 자동 하이라이트 되는 것을 방지하기 위해 Console 객체의 print 메서드 사용
            Console().print(colored_text, highlight=False)

            ########################################## 여기까지가 변경된 부분
        
        whole_data += dict_data
        
    for datapoint in whole_data:
        for key, value in datapoint.items():
            if value == "np.nan":
                datapoint[key] = np.nan
    
    data_final = pd.DataFrame(whole_data)

    if len(data_final)==0:
        print("No data")
        return None
    else:
        # iupac name check

        if len(false_page_idx)==0:
            print('Nice')
            print('Every IUPAC is correct')
            
        else:
            richprint('Wrong IUPAC names')
            richprint(false_page_idx)

            false_iupac_compare(short_text, data_final, false_page_idx)

            

        return data_final, false_page_idx

# Extract data

In [ ]:
# Enter bibliographic information of the paper to extract information from

# example 
# title='Pd(II)-Catalyzed Aminoacetoxylation of Alkenes Via Tether Formation'
# doi='https://pubs.acs.org/doi/10.1021/acs.orglett.2c01838'
# publish='2022.24.28'

title=
doi=
publish=


In [ ]:
# Enter the file name inside DL_TLC_project/LLM_data_extractor/SI

data_final, false_page_idx = extract_data_GPT('10.pdf')

In [ ]:
# delete wrong

del_list = []
for arr in list(false_page_idx.values()):
    del_list += list(arr)
del_list

del_method = input("Do you want to use del list? (y/n): ")
if del_method == 'y':
    user_input = input("Enter the correct indices to exclude from the delete list (separate by space): ")
    if user_input == "":
        delete_final = del_list.copy()
    else:
        exe = [float(x) for x in user_input.split()]
        delete_final = [i for i in del_list if i not in exe]
    # 추가로 삭제할 인덱스 받기
    add_user_input = input("Enter additional indices to delete (press Enter if none): ")
    if add_user_input != "":
        add_delete = [float(x) for x in add_user_input.split()]
        delete_final += add_delete
    # 중복 제거
    delete_final = list(set(delete_final))
    data_final.drop(delete_final, axis=0, inplace=True)
    print(f'{delete_final} is deleted: length = {len(data_final)}')
elif del_method == 'n': 
    user_input = input("Enter incorrect indices (separated by spaces): ")
    numbers = [float(x) for x in user_input.split()]
    data_final.drop(numbers, axis=0, inplace=True)
    print(f'{numbers} is in delete list: length = {len(data_final)}')



# data_final.index = range(len(data_final))

In [ ]:
# delete nan

data_final=data_final[np.all(~pd.isna(data_final[['compound IUPAC name','eluent1','ratio','Rf']]), axis=1)]
data_final.index = range(len(data_final))
len(data_final)

In [ ]:
# 중복제거
data_id = ['_'.join(map(str,data_final.iloc[i,[1,2,3,4,5]]))for i in range(len(data_final))]
data_final.loc[:,'data_ID'] = data_id
data_final = data_final.groupby(by = 'data_ID', as_index = False).agg({'compound name':'first',
                                                                      'compound IUPAC name' : 'first',
                                                                      'eluent1' : 'first',
                                                                      'eluent2' : 'first',
                                                                      'ratio' : 'first',
                                                                      'Rf' : 'first'})
data_final = data_final.reset_index(drop=True)
len(data_final)

# IUPAC name --> SMILES

In [ ]:
import subprocess

def iupac_to_smiles_local_opsin(iupac_name):
    jar_path = 'opsin-cli-2.8.0-jar-with-dependencies.jar'  # 본인 경로
    command = ['java', '-jar', jar_path]
    
    result = subprocess.run(command, input = iupac_name, capture_output=True, text=True)

    if result.returncode == 0:
        smiles = result.stdout.strip()
        if smiles:
            return smiles
        else:
            return None
    else:
        raise RuntimeError(f"OPSIN failed: {result.stderr}")

        


In [ ]:
# compound: IUPAC --> smiles

smiles_list = []
for iupac in tqdm(data_final['compound IUPAC name']):
    if iupac:
        smiles_list.append(iupac_to_smiles_local_opsin(iupac))
    else:
        smiles_list.append(np.nan)
        
data_final2 = data_final.copy(deep=True)
data_final2['compound IUPAC name'] = smiles_list
data_final2.rename(columns={'compound IUPAC name': 'Smiles'}, inplace=True)
data_final2=data_final2[pd.notna(data_final2['Smiles'])]
data_final2.reset_index(drop=True, inplace=True)
data_final2

# Save

In [ ]:
import os

folder_path = "data_extracted_gpt"
file_count = len([
    f for f in os.listdir(folder_path)
    if os.path.isfile(os.path.join(folder_path, f))
])

richprint(f"📁 '{folder_path}' 폴더에 있는 파일 개수: {file_count}")

# gpt -data 개수 확인
data_num=[]
for i in np.arange(1,file_count+1,1):
    num = len(pd.read_csv(f'./data_extracted_gpt/datatable{i}.csv'))
    data_num.append(num)
richprint(f'논문 별 평균 데이터 수: {np.mean(data_num)}')
richprint(f'gpt가 모은 데이터 수: {np.sum(data_num)}')

In [ ]:
if doi ==last_doi:
    print('Check doi')
else:
    data_final3= data_final2[~pd.isna(data_final2['Smiles'])].copy()
    data_final3['title'] = [title for i in range(len(data_final3))]
    data_final3['doi'] = [doi for i in range(len(data_final3))]
    data_final3['publish'] = [publish for i in range(len(data_final3))]
    data_final3.index = range(len(data_final3))
    data_final3
data_final3

In [134]:
# 최종 저장
data_final3.to_csv(f'data_extracted_gpt/datatable{file_count+1}.csv', index=False)